# Data Cleaning — TheLook Customer Analysis

**Scope:** Clean the 4 tables used for Customer Analysis.

**Input:** `../data/raw/` → **Output:** `../data/processed/`

| Table | Key cleaning steps |
|---|---|
| `order_items` | Parse datetimes, drop `inventory_item_id` |
| `orders` | Parse datetimes |
| `users` | Parse datetimes, drop PII columns, fix country names |
| `products` | Drop `name`, `sku`, `distribution_center_id`; fill brand nulls |

## 0. Imports & Load Data

In [ ]:
import os
import pandas as pd
import numpy as np

RAW       = os.path.join("..", "data", "raw")
PROCESSED = os.path.join("..", "data", "processed")

order_items = pd.read_csv(os.path.join(RAW, "order_items.csv"))
orders      = pd.read_csv(os.path.join(RAW, "orders.csv"))
products    = pd.read_csv(os.path.join(RAW, "products.csv"))
users       = pd.read_csv(os.path.join(RAW, "users.csv"))

In [ ]:
# Quick overview
for name, df in [("order_items", order_items), ("orders", orders),
                 ("products", products), ("users", users)]:
    print(f"{name:<15} {df.shape[0]:>9,} rows  x  {df.shape[1]} cols")

## 1. Table: order_items

In [ ]:
order_items.info()

In [ ]:
# Parse datetimes
for col in ['created_at', 'shipped_at', 'delivered_at', 'returned_at']:
    order_items[col] = pd.to_datetime(order_items[col], format='mixed', utc=True)

# Drop column not needed for analysis
order_items = order_items.drop(columns=['inventory_item_id'])

In [ ]:
print("Missing values:")
print(order_items.isna().sum()[order_items.isna().sum() > 0].to_string() or "  none")
print(f"Duplicates: {order_items.duplicated().sum()}")
print("\nStatus values:", order_items['status'].unique())
print("\nSale price:")
print(order_items['sale_price'].describe().round(2))

In [ ]:
order_items.to_csv(os.path.join(PROCESSED, "order_items_clean.csv"), index=False)
print("✅ order_items_clean.csv saved")

## 2. Table: orders

In [ ]:
orders.info()

In [ ]:
# Parse datetimes
for col in ['created_at', 'returned_at', 'shipped_at', 'delivered_at']:
    orders[col] = pd.to_datetime(orders[col], format='mixed', utc=True)

In [ ]:
print("Missing values (%)")
print(orders.isna().mean().mul(100).round(2).to_string())
print(f"\nDuplicates: {orders.duplicated().sum()}")
print("\nStatus values:", orders['status'].unique())
print("num_of_item:", orders['num_of_item'].describe().round(2).to_dict())

In [ ]:
orders.to_csv(os.path.join(PROCESSED, "orders_clean.csv"), index=False)
print("✅ orders_clean.csv saved")

## 3. Table: users

In [ ]:
users.info()

In [ ]:
# Parse datetime
users['created_at'] = pd.to_datetime(users['created_at'], format='mixed', utc=True)

# Drop PII and columns not needed for customer analysis
users = users.drop(columns=['first_name', 'last_name', 'email',
                             'street_address', 'postal_code',
                             'latitude', 'longitude'])

# Fix country name inconsistencies
users['country'] = users['country'].replace({
    'España': 'Spain',
    'Deutschland': 'Germany'
})

In [ ]:
print("Missing values:")
print(users.isna().sum()[users.isna().sum() > 0].to_string() or "  none")
# Note: city nulls are small — analysis uses country/state level, so acceptable
print(f"\nDuplicates: {users.duplicated().sum()}")
print("\nAge:", users['age'].describe().round(2).to_dict())
print("Gender:", users['gender'].value_counts().to_dict())
print("Traffic source:", users['traffic_source'].unique())

In [ ]:
users.to_csv(os.path.join(PROCESSED, "users_clean.csv"), index=False)
print("✅ users_clean.csv saved")

## 4. Table: products

In [ ]:
products.info()

In [ ]:
# Drop columns not needed for analysis
products = products.drop(columns=['name', 'sku', 'distribution_center_id'])

# Fill brand nulls
products['brand'] = products['brand'].fillna('Unknown')

In [ ]:
print("Missing values:")
print(products.isna().sum()[products.isna().sum() > 0].to_string() or "  none")
print(f"\nDuplicates: {products.duplicated().sum()}")
print("\nCategories:", products['category'].nunique(), "unique")
print("Departments:", products['department'].unique())
print("\nPrice stats:")
print(products[['cost', 'retail_price']].describe().round(2))

In [ ]:
products.to_csv(os.path.join(PROCESSED, "products_clean.csv"), index=False)
print("✅ products_clean.csv saved")

---

## Summary

All 4 tables cleaned and saved to `../data/processed/`:

| File | Key changes |
|---|---|
| `order_items_clean.csv` | Datetimes parsed, `inventory_item_id` dropped |
| `orders_clean.csv` | Datetimes parsed |
| `users_clean.csv` | PII dropped, country names standardised |
| `products_clean.csv` | Unused columns dropped, brand nulls filled |